In [1]:
import numpy as np
import os
import sys
from stable_baselines3 import DQN
import requests
from IPython.display import clear_output
import pandas as pd

In [2]:
sys.path.insert(0,'../boptestGym')
from boptestGymEnv import BoptestGymEnv
from boptestGymEnv import NormalizedObservationWrapper
from boptestGymEnv import DiscretizedActionWrapper

In [3]:
url = "http://localhost:80"

In [4]:
class BoptestGymEnvCustomReward(BoptestGymEnv):
    def get_reward(self):
        # Compute BOPTEST core kpis
        kpis = requests.get('{0}/kpi/{1}'.format(self.url, self.testid)).json()['payload']

        ener_coef = 1.0
        temp_coef = 1.0
        
        # todo: search for best reward function
        reward = - ((kpis['tdis_tot']*temp_coef) + (kpis['ener_tot']*ener_coef))

        # Record current objective integrand for next evaluation
        self.objective_integrand = reward
        return reward

In [5]:
# tested period: nov 1 to december 31 paper settings
def create_env():
    env = BoptestGymEnvCustomReward(url   = url,
                    testcase              = 'bestest_hydronic_heat_pump',
                    actions               = ['oveHeaPumY_u'],
                    observations          = {'reaTZon_y':(273.,324.),
                                             'weaSta_reaWeaTDryBul_y':(257.,305.),
                                             'weaSta_reaWeaHDirNor_y':(0.,862.)
                                            },
                    random_start_time     = False,
                    start_time            = 305*24*3600, #nov 1 like the paper
                    max_episode_length    = 61 * 24*3600, #2 month testing period
                    warmup_period         = 24*3600,
                    predictive_period     = 0,
                    regressive_period     = 4*1800,
                    step_period           = 1800
                    )
    env = NormalizedObservationWrapper(env)
    env = DiscretizedActionWrapper(env,n_bins_act=1)
    return env

In [6]:
data_regime = 6

for data_regime in [6, 27]:
    for training_episodes in [25, 50]:
        for planning_steps in [50, 100]:
            for mod in ["DynaPINN", "DynaNN", "DynaRC"]:
                model= DQN.load(f"Saved Models/{mod}_{data_regime}_{training_episodes}_{planning_steps}steps")
                env = create_env()
                done = False
                obs, _ = env.reset()
                rows = []

                i=0
                while i<=2928:
                    # Clear the display output at each step
                    clear_output(wait=True)
    
                    # Compute control signal
                    action, _ = model.predict(obs, deterministic=True)
                    kpis = requests.get('{0}/kpi/{1}'.format(url, env.testid)).json()['payload']

                    if isinstance(action, (tuple, list, np.ndarray)):
                        action = int(np.array(action).flatten()[0])
                    # Print the current operative temperature and decided action
                    print('-------------------------------------------------------------------')
                    print(f"{mod}_{data_regime}_{training_episodes}_{planning_steps}")
                    print("obs: %s"%obs)
                    print("act: %s"%action)
                    print("%s /2928"%i)
                    print('-------------------------------------------------------------------')
                    i+=1
                    # Implement action
                    rows.append({
                        "T_zone": obs[0],
                        "t_out": obs[1],
                        "Psol_Wm2": obs[2],
                        "action": action,
                        "energy_kWh": kpis["ener_tot"],
                        "discomfort": kpis["tdis_tot"]
                    })
                    obs, reward, terminated, truncated, info = env.step(action)  # send the action to the environment
                    done = (terminated or truncated)

                df = pd.DataFrame(rows)
                df.to_csv(f"Testing/{mod}_{data_regime}_{training_episodes}_{planning_steps}steps.csv", index=False)
                env.stop()
                del model

KeyboardInterrupt: 

In [8]:
env.stop()